# Getting Started with cleanskate

**cleanskate** is a Python package to access clean figure skating data. Results are parsed from ISU protocols and stored on the cloud, and downloaded

The main 
To load data, import the `Dataset` handler from `cleanskate. The object supports methods for loading events, overall results, elements, and program component scores.

In [ ]:
import pandas as pd

from cleanskate import Dataset

ds = Dataset()

events = ds.load_events()
results = ds.load_results()
segments = ds.load_segments()
standings = ds.load_standings()
elements = ds.load_elements()
components = ds.load_program_components()

print(f'Events: {len(events)}')
print(f'Segments: {len(segments)}')
print(f'Results: {len(results)}')
print(f'Participants: {len(results["name"].unique())}')
print(f'Elements: {len(elements)}')


Events: 93
Segments: 798
Results: 11741
Participants: 1433
Elements: 100732


## Standings


## Skater performance per element

We can subset the elements data frame to identify who gets the highest scores on particular elements. Let's start by focusing on the triple axel.

In [5]:
import numpy as np

elements_3a = ds.load_elements(attempt_code='3A')
skater_scores_3a = elements_3a.groupby('name', as_index=False).agg(
    count=('goe', len),
    mean_goe=('goe', lambda x: sum(x)/len(x)),
    prob_clean=('info_flags', lambda x: np.mean(pd.isna(x))),
    prob_fall=('info_flags', lambda x: np.mean(~pd.isna(x) & x.str.contains('F')))
    )

skater_scores_3a[ skater_scores_3a['count'] >= 10].sort_values(by='mean_goe', ascending=False).head(10)

elements_3a.groupby('element_code').size()

element_code
3A          2220
3A+REP        43
3A<          265
3A<+REP       22
3A<<          70
3A<<+REP       5
3Aq          206
3Aq+REP       17
dtype: int64

## Best elements for a skater

Another way to dice this data is to examine which elements generally get the most points for a skater. We can use the filtering in `load_elements` to subset to 

Let's try Ilia Malinin for an example.

In [19]:
import matplotlib as mpl
import seaborn as sns

mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False

senior_jumps = ds.load_elements(event_level="Senior", element_family=["Jump", "Jump Combo", "Jump Sequence"])

ilia_elements = senior_jumps[ senior_jumps['name'] == 'Ilia MALININ' ].copy()
jump_counts = ilia_elements.groupby(['element_family', 'attempt_code'], as_index=False).agg(
    count=('goe', len),
    mean_goe=('goe', np.mean),
    mean_points=('panel_score', np.mean),
    prob_clean=('info_flags', lambda x: np.mean(pd.isna(x))),
    prob_fall=('info_flags', lambda x: np.mean(~pd.isna(x) & x.str.contains('F')))
    ).sort_values(by='mean_points', ascending=False)  

jump_counts = jump_counts[ jump_counts['count'] >= 5].sort_values(by='mean_points', ascending=False)
jump_counts

,element_family,attempt_code,count,mean_goe,mean_points,prob_clean,prob_fall
31,Jump Sequence,4S+3A+SEQ,6,0.938333,20.408333,0.833333,0.000000
19,Jump Combo,4Lz+3T,12,3.070833,19.606667,0.916667,0.000000
29,Jump Sequence,3Lz+3A+SEQ,5,1.578000,16.868000,1.000000,0.000000
27,Jump Combo,4T+3T,11,1.407273,16.228182,0.909091,0.000000
11,Jump,4Lz,14,3.557143,15.057143,1.000000,0.000000
9,Jump,4F,20,3.312000,14.312000,0.900000,0.000000
8,Jump,4A,5,1.002000,13.502000,0.600000,0.000000
14,Jump,4T,9,1.898889,11.504444,0.888889,0.111111
13,Jump,4S,8,1.652500,10.677500,0.750000,0.000000
4,Jump,3A,26,2.230385,10.445769,1.000000,0.000000


In [ ]:
elements_to_keep = jump_counts['attempt_code'].values
plot_data = ilia_elements[ ilia_elements['attempt_code'].isin(elements_to_keep) ]

mpl.pyplot.figure(figsize=(10, 5))
ax = sns.boxplot(
    plot_data,
    x='attempt_code',
    y='panel_score',
    order=elements_to_keep,
    width=0.6,
    fliersize=3,            # smaller outliers
    linewidth=1.2,
    color="#899BB8"
    )

ax.set_xlabel("Attempt", fontsize=12)
ax.set_ylabel("Panel score", fontsize=12)
ax.tick_params(axis='x', rotation=30)

In [4]:
senior_jumps[ senior_jumps['attempt_code'].str.startswith('4A') ]

,event_label,event_series,event_level,segment_label,name,noc,element_number,element_code,attempt_code,element_family,valid_element,clean_element,info_flags,base_value,second_half_bonus,goe,panel_score,judge_scores
13512,Grand Prix Espoo 2022,Grand Prix,Senior,Men FS,Ilia MALININ,USA,1,4Aq,4A,Jump,True,False,q,12.5,False,-3.75,8.75,"[-4, -2, -2, -3, -2, -3, -5, -4, -3]"
18914,GP Rostelecom Cup 2018,Grand Prix,Senior,Men FS,Artur DMITRIEV,RUS,2,4A<<,4A,Jump,True,False,<<,8.0,False,-4.00,4.00,"[-5, -5, -5, -5, -5, -5, -5, -5, -5]"
20355,GP Skate America 2022,Grand Prix,Senior,Men FS,Ilia MALININ,USA,1,4A,4A,Jump,True,True,NaN,12.5,False,4.11,16.61,"[4, 2, 2, 3, 3, 3, 4, 4, 4]"
21514,Olympics 2022,Olympics,Senior,Men FS,Yuzuru HANYU,JPN,1,4A<,4A,Jump,True,False,<,10.0,False,-5.00,5.00,"[-5, -5, -5, -5, -5, -5, -5, -5, -5]"
24707,Worlds 2023,Worlds,Senior,Men FS,Ilia MALININ,USA,1,4A,4A,Jump,True,True,NaN,12.5,False,0.36,12.86,"[0, 3, 1, -1, 0, 0, 0, 1, 0]"
25310,Worlds 2024,Worlds,Senior,Men FS,Ilia MALININ,USA,1,4A,4A,Jump,True,True,NaN,12.5,False,3.93,16.43,"[3, 1, 4, 2, 3, 3, 4, 3, 5]"
25933,Worlds 2025,Worlds,Senior,Men FS,Ilia MALININ,USA,2,4Aq,4A,Jump,True,False,q,12.5,False,0.36,12.86,"[0, 1, 0, 0, 0, -2, 2, 1, 0]"
